In [1]:
USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — Gauged Minimal Guidance

## Objective
Maximize the **Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** on the out-of-sample evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (Gauged)**: All 531 basins have training streamflow observations `QOBS_TRAIN` (`mm/day`) available during the weights training period (`2003-10-01` to `2008-09-30`, 5 water years, `1,827` days).
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`). The final 365 days (`2002-10-01` to `2003-09-30`) also serve as the 1-year meteorological spin-up prior to `2003-10-01`.
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period (`QOBS_TRAIN` observed).
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Modeling & Exploration Scope
You are free to explore any machine learning architecture, sequence length, feature representation, normalization strategy, loss function, regularization, optimizer schedule, or multi-model ensembling strategy, provided all predictions strictly respect the causal temporal masks and spatial fold boundaries.

In [2]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "gauged"
print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}).")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (gauged): 531 basins, 3653 days (1998-10-01 to 2008-09-30).
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [3]:
# Begin Stage

In [4]:
INSTRUCTION = [
    "Design a higher-capacity multi-scale sequence architecture (such as a stacked residual LSTM/GRU, dilated causal Temporal Convolutional Network + recurrent hybrid, or multi-timescale recurrent network) with engineered temporal context features (rolling exponentially weighted moving averages over multiple lookback windows, first differences, and interaction terms across the 3 meteorological forcing products) and train a multi-seed ensemble.",
    "Explore a multi-head or gated mixture-of-experts sequence model where static catchment attributes condition the network across all layers (e.g., via projection gates or feature-wise linear modulation), combined with cosine annealing learning rate schedule, gradient clipping, and weight decay.",
    "Implement a dual-pathway or multi-resolution recurrent architecture that processes long-term 365-day seasonal/interannual context alongside high-resolution recent 30-to-90-day dynamics, using basin-normalized NSE loss with curriculum or multi-objective huber/log-MSE regularization.",
    "Combine diverse sequence model families (e.g., gated recurrent networks, causal depthwise-separable convolutions, and attention pooling over historical states) into a blended multi-seed ensemble with rich static-dynamic cross-features.",
    "Explore input representation learning (learned nonlinear embeddings of the 3 forcing products Daymet, Maurer, and NLDAS plus static catchment embeddings) and output post-processing (non-negative softplus/ELU activation and learned multi-lag autoregressive/moving-average smoothing filters on predicted discharge).",
    "Optimize training dynamics and generalization: sample more sequences per basin, apply feature dropout or Gaussian input noise regularization, use Stochastic Weight Averaging (SWA) or exponential moving average (EMA) of model weights over the final epochs, and ensemble multiple independent training runs.",
    "Explore an Entity-Aware / input-modulated recurrent architecture with residual highway connections, layer normalization, and a multi-horizon or auxiliary loss head that improves representation quality across all 531 basins.",
    "Synthesize the strongest architectural components, feature transformations, loss formulations, and ensembling strategies discovered so far into a cohesive, high-accuracy multi-model ensemble within the runtime budget."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [5]:
# [rewrite_cell]
# ==============================================================================
# ENSEMBLE EA-LSTM WITH TEMPORAL CONTEXT & DUAL-OBJECTIVE NSE LOSS
# ==============================================================================

# Configuration
SEQ_LEN = 365
HIDDEN_SIZE = 320
NUM_LAYERS = 2
DROPOUT = 0.4
BATCH_SIZE = 128
LR = 1.2e-3
EPOCHS = 70
SAMPLES_PER_BASIN_PER_EPOCH = 25
NSE_STABILITY_EPS = 0.05 

class EntityAwareLSTM(nn.Module):
    def __init__(self, dyn_dim, stat_dim, hidden_size, num_layers=2, dropout=0.4):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        
        # Deeper static embedding for initialization and context
        self.stat_embed = nn.Sequential(
            nn.Linear(stat_dim, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Dropout(dropout)
        )
        
        self.init_h = nn.Linear(hidden_size, num_layers * hidden_size)
        self.init_c = nn.Linear(hidden_size, num_layers * hidden_size)
        
        # Concatenate dynamic inputs with static context at each step
        self.lstm = nn.LSTM(dyn_dim + hidden_size, hidden_size, num_layers=num_layers, 
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, 1)
        )

        for name, param in self.lstm.named_parameters():
            if "weight" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0.0)
                # Forget gate bias initialization for long-term dependencies
                n = param.size(0)
                param.data[n // 4 : n // 2].fill_(4.0) 

    def forward(self, x_dyn, x_stat):
        B, L, _ = x_dyn.shape
        s_emb = self.stat_embed(x_stat)
        h0 = self.init_h(s_emb).view(self.num_layers, B, self.hidden_size).contiguous()
        c0 = self.init_c(s_emb).view(self.num_layers, B, self.hidden_size).contiguous()
        
        s_emb_seq = s_emb.unsqueeze(1).expand(-1, L, -1)
        x_combined = torch.cat([x_dyn, s_emb_seq], dim=-1)
        
        out, _ = self.lstm(x_combined, (h0, c0))
        return self.head(out)

# 1. Feature Engineering
def get_rolling(x, w):
    res = np.zeros_like(x)
    cumsum = np.cumsum(x, axis=1)
    res[:, w:] = (cumsum[:, w:] - cumsum[:, :-w]) / w
    res[:, :w] = cumsum[:, :w] / np.arange(1, w + 1)
    return res

dates_dt = pd.to_datetime(DATES)
d_sin = np.sin(2 * np.pi * dates_dt.dayofyear.values / 365.25).astype(np.float32)
d_cos = np.cos(2 * np.pi * dates_dt.dayofyear.values / 365.25).astype(np.float32)
doy_feats = np.stack([np.tile(d_sin, (len(BASINS), 1)), np.tile(d_cos, (len(BASINS), 1))], axis=-1)

p_idx = [0, 5, 10]
p_mean = DYNAMIC_INPUTS[..., p_idx].mean(axis=-1)
p_7d = get_rolling(p_mean, 7)
p_30d = get_rolling(p_mean, 30)

X_dyn_raw = np.concatenate([
    DYNAMIC_INPUTS, AUX_FORCINGS, doy_feats,
    np.expand_dims(p_mean, -1),
    np.expand_dims(p_7d, -1),
    np.expand_dims(p_30d, -1),
    DYNAMIC_INPUTS[..., [2, 3, 7, 8, 12, 13]].mean(axis=-1, keepdims=True), # T_mean
    DYNAMIC_INPUTS[..., [1, 6, 11]].mean(axis=-1, keepdims=True),           # SR_mean
    DYNAMIC_INPUTS[..., [4, 9, 14]].mean(axis=-1, keepdims=True),           # VP_mean
    np.log1p(np.maximum(p_mean[..., None], 0))
], axis=-1)

X_stat_raw = np.nan_to_num(STATIC_ATTRIBUTES, nan=0.0)

# Normalization
d_mean = X_dyn_raw[:, ERA_TRAIN_MASK, :].mean(axis=(0, 1), keepdims=True)
d_std = X_dyn_raw[:, ERA_TRAIN_MASK, :].std(axis=(0, 1), keepdims=True) + 1e-6
X_dyn_norm = ((X_dyn_raw - d_mean) / d_std).astype(np.float32)

s_mean = X_stat_raw.mean(axis=0, keepdims=True)
s_std = X_stat_raw.std(axis=0, keepdims=True) + 1e-6
X_stat_norm = ((X_stat_raw - s_mean) / s_std).astype(np.float32)

y_mean = float(np.nanmean(QOBS_TRAIN[:, ERA_TRAIN_MASK]))
y_std = float(np.nanstd(QOBS_TRAIN[:, ERA_TRAIN_MASK])) + 1e-6
Y_norm = ((QOBS_TRAIN - y_mean) / y_std).astype(np.float32)

per_basin_vars = np.ones(len(BASINS), dtype=np.float32)
for b in range(len(BASINS)):
    vb = QOBS_TRAIN[b, ERA_TRAIN_MASK]
    vb = vb[~np.isnan(vb)]
    if len(vb) > 10: per_basin_vars[b] = np.var(vb / y_std)

# Prepare GPU tensors
X_d_gpu = torch.from_numpy(X_dyn_norm).to(DEVICE)
X_s_gpu = torch.from_numpy(X_stat_norm).to(DEVICE)
Y_gpu = torch.from_numpy(Y_norm).to(DEVICE)
B_var_gpu = torch.from_numpy(per_basin_vars).to(DEVICE)
TRAIN_MASK_GPU = torch.from_numpy(ERA_TRAIN_MASK).to(DEVICE)
seq_offsets = torch.arange(SEQ_LEN, device=DEVICE).unsqueeze(0)

def train_model(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = EntityAwareLSTM(X_dyn_norm.shape[-1], X_stat_norm.shape[-1], HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
    steps_per_epoch = math.ceil((len(BASINS) * SAMPLES_PER_BASIN_PER_EPOCH) / BATCH_SIZE)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=steps_per_epoch, epochs=EPOCHS)
    scaler = torch.amp.GradScaler("cuda")
    
    train_idx = np.where(ERA_TRAIN_MASK)[0]
    t_min, t_max = train_idx[0] - SEQ_LEN + 32, train_idx[-1] - SEQ_LEN
    
    for epoch in range(EPOCHS):
        model.train()
        p_b = np.repeat(np.arange(len(BASINS)), SAMPLES_PER_BASIN_PER_EPOCH)
        p_t = np.random.randint(t_min, t_max + 1, size=len(p_b))
        perm = np.random.permutation(len(p_b))
        p_b, p_t = p_b[perm], p_t[perm]

        for i in range(0, len(p_b), BATCH_SIZE):
            idx_b, idx_t = torch.from_numpy(p_b[i:i+BATCH_SIZE]).to(DEVICE), torch.from_numpy(p_t[i:i+BATCH_SIZE]).to(DEVICE)
            win_t = idx_t.unsqueeze(1) + seq_offsets
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                y_hat = model(X_d_gpu[idx_b.unsqueeze(1), win_t], X_s_gpu[idx_b]).squeeze(-1)
                y_true = Y_gpu[idx_b.unsqueeze(1), win_t]
                m = TRAIN_MASK_GPU[win_t] & (~torch.isnan(y_true))
                if m.any():
                    # Dual-objective loss: Weighted MSE + Weighted LogMSE
                    nse_loss = torch.mean(((y_hat[m] - y_true[m])**2) / (B_var_gpu[idx_b].unsqueeze(1).expand(-1, SEQ_LEN)[m] + NSE_STABILITY_EPS))
                    y_true_unnorm = torch.clamp(y_true[m] * y_std + y_mean, min=0.0)
                    y_hat_unnorm = torch.clamp(y_hat[m] * y_std + y_mean, min=0.0)
                    log_loss = torch.mean((torch.log1p(y_hat_unnorm) - torch.log1p(y_true_unnorm))**2)
                    loss = nse_loss + 0.1 * log_loss
                    
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
            scheduler.step()
    return model

# 3-Seed Ensemble for Stability
seeds = [42, 88, 2024]
Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)
for s in seeds:
    print(f"Training Model (Seed {s})...")
    m = train_model(s)
    m.eval()
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for b_start in range(0, len(BASINS), 64):
            b_end = min(b_start + 64, len(BASINS))
            y_hat = m(X_d_gpu[b_start:b_end], X_s_gpu[b_start:b_end]).squeeze(-1)
            y_mm = torch.clamp(y_hat * y_std + y_mean, min=0.0)
            Q_PRED[b_start:b_end] += (y_mm.cpu().numpy() / len(seeds))

print("Ensemble inference complete.")

Training Model (Seed 42)...


/tmp/ipykernel_23/1154373012.py:168: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Training Model (Seed 88)...
Training Model (Seed 2024)...
Ensemble inference complete.


In [6]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (gauged | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.82411 | Mean NSE: 0.69865 | IQR: [0.71226, 0.87324] | Basins < 0: 22
Candidate metric to be maximized: 0.824109


In [7]:
# End Stage